# Day 6: Publish Your Agent

This notebook documents the journey from Day 4-5 notebooks to a deployed production agent on Streamlit Cloud.

## What We Built

Starting from Day 4 notebooks with inline code, we:
1. Extracted code into standalone Python modules in `app/` folder
2. Created CLI interface for local testing
3. Built Streamlit web UI with streaming responses
4. Deployed to Streamlit Cloud with secrets management

**Deployed Agent:** https://aihero-nbzqjktqedjuiq6bqjbjwn.streamlit.app/

## Learning Objectives

1. Understand notebook-to-production refactoring patterns
2. Learn module extraction with dependency injection
3. Implement CLI and web UI interfaces
4. Deploy to cloud with environment-based configuration
5. Compare learning vs production architectures

## Table of Contents

1. **Architecture Overview** - From notebooks to app/ modules
2. **Module Extraction Patterns** - ingest.py, search_tools.py, search_agent.py, logs.py
3. **SearchTool Class Pattern** - Dependency injection vs global variables
4. **CLI Interface** - Interactive loop with asyncio
5. **Streamlit Web UI** - Chat interface with streaming
6. **Deployment Process** - Streamlit Cloud step-by-step
7. **Learnings Summary** - Notebook vs Production comparison

---

## 1. Architecture Overview

### From Notebooks to Production Modules

**Day 4 Notebook Architecture:**
```
course/day4.ipynb:
  - Data loading (read_repo_data, chunk_sliding_window)
  - Search index (text_index global variable)
  - text_search function (uses global text_index)
  - Agent definition (pydantic_agent with @tool_plain decorator)
  - Inline execution (cells run top-to-bottom)
```

**Production Architecture (app/ folder):**
```
app/
├── pyproject.toml          # uv project with dependencies
├── requirements.txt        # Generated from uv.lock (170 packages)
├── ingest.py              # Data loading + chunking + indexing
├── search_tools.py        # SearchTool class (encapsulates index)
├── search_agent.py        # init_agent factory function
├── logs.py                # Interaction logging with environment config
├── main.py                # CLI entry point
└── app.py                 # Streamlit web UI entry point
```

### Key Architectural Changes

| Aspect | Notebook (Learning) | Production (app/) |
|--------|---------------------|-------------------|
| **Index Storage** | Global `text_index` variable | `SearchTool` class with encapsulated index |
| **Agent Creation** | Direct instantiation in cell | `init_agent(search_tool)` factory function |
| **Configuration** | Hardcoded DataTalksClub/faq | Environment variables (OPENAI_API_KEY) |
| **Execution** | Cell-by-cell (exploratory) | Entry points (main.py, app.py) |
| **Dependencies** | Inline in notebook | uv project with hash-pinned requirements |

### Why Refactor?

1. **Reusability**: Modules can be imported by CLI, web UI, and future interfaces
2. **Testability**: Functions with clear inputs/outputs are easier to test
3. **Maintainability**: Changes to search logic don't require editing entry points
4. **Deployability**: Streamlit Cloud runs Python modules, not notebooks
5. **Type Safety**: Type hints + docstrings enable IDE autocomplete and static analysis

---

## 2. Module Extraction Patterns

Each module follows a clear separation of concerns pattern.

### ingest.py - Data Loading Pipeline

**Purpose:** Load FAQ data from GitHub, chunk documents, build search index.

**Key Functions:**
- `read_repo_data(owner: str, repo: str) -> list[dict]` - GitHub API data fetching
- `chunk_documents(docs: list[dict]) -> list[dict]` - Sliding window chunking
- `index_data(chunks: list[dict]) -> Index` - Build minsearch index

**Pattern:** Pipeline of pure functions (no side effects, no global state).

```python
# Usage pattern
docs = read_repo_data("DataTalksClub", "faq")
chunks = chunk_documents(docs)
index = index_data(chunks)
```

**Learning → Production:**
- Notebook: Functions imported from day1.py, day2.py
- Production: Self-contained module with all dependencies

### search_tools.py - SearchTool Class

**Purpose:** Encapsulate search index for dependency injection into agent.

**Why a class?** Day 4 notebook used global `text_index` variable. This works for learning but creates problems:
- **Testing:** Hard to mock or swap implementations
- **Multiple Indexes:** Can't have course FAQ + OWASP FAQ simultaneously
- **Configuration:** Can't parameterize index settings

**SearchTool Class Pattern:**

```python
class SearchTool:
    """Encapsulates search index for agent tool registration.
    
    Pattern: Dependency injection via __init__ (not global variable).
    Benefits:
    - Testable: Mock index for unit tests
    - Reusable: Multiple SearchTool instances for different indexes
    - Configurable: Pass different boost_dict, num_results
    """
    
    def __init__(self, index: Index):
        """Initialize with minsearch Index instance.
        
        Args:
            index: Pre-built minsearch.Index from ingest.index_data()
        """
        self.index = index
    
    def search(self, query: str, num_results: int = 5) -> list[dict]:
        """Search FAQ using encapsulated index.
        
        This method will be registered as agent tool via @agent.tool_plain.
        """
        results = self.index.search(
            query=query,
            boost_dict={"title": 2.0, "content": 1.0},
            num_results=num_results
        )
        return [
            {
                "title": r.get("title", "Untitled"),
                "content": r.get("content", "")[:500],
                "score": r.get("score", 0.0)
            }
            for r in results
        ]
```

**Usage Pattern (Dependency Injection):**

```python
# 1. Build index
index = index_data(chunks)

# 2. Inject into SearchTool
search_tool = SearchTool(index)

# 3. Pass to agent initialization
agent = init_agent(search_tool)
```

**Key Insight:** The index is created ONCE during initialization, then passed to SearchTool, then to agent. No global variables. Clean dependency flow.

### search_agent.py - Agent Factory

**Purpose:** Create configured Pydantic AI agent with search tool.

**Why a factory function?** Agent initialization requires:
1. System prompt template with repo info
2. Tool registration via @agent.tool_plain decorator
3. SearchTool dependency injection

**init_agent Factory Pattern:**

```python
SYSTEM_PROMPT_TEMPLATE = """You are an FAQ assistant for the {repo_owner}/{repo_name} repository.

Rules:
1. ALWAYS use search tool before answering
2. Base answers ONLY on search results
3. Cite sources explicitly
4. If no results, say "I don't have information about that"
"""

def init_agent(
    search_tool: SearchTool,
    repo_owner: str = "DataTalksClub",
    repo_name: str = "faq"
) -> Agent:
    """Initialize FAQ agent with search tool and system prompt.
    
    Args:
        search_tool: SearchTool instance with encapsulated index
        repo_owner: GitHub repository owner (default: DataTalksClub)
        repo_name: GitHub repository name (default: faq)
    
    Returns:
        Configured pydantic-ai Agent ready for run() or run_stream()
    """
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        repo_owner=repo_owner, repo_name=repo_name
    )
    
    agent = Agent('openai:gpt-4o-mini', system_prompt=system_prompt)
    
    @agent.tool_plain
    def search_faq(query: str, num_results: int = 5) -> list[dict]:
        """Search FAQ using text search.
        
        Args:
            query: Search query extracted from user's question
            num_results: Maximum results to return (default: 5)
        
        Returns:
            List of dicts with title, content, score
        """
        return search_tool.search(query, num_results)
    
    return agent
```

**Pattern Insights:**

1. **Parameterized System Prompt**: Template with format() allows different repos
2. **Decorator Inside Factory**: @agent.tool_plain must be inside function to capture search_tool via closure
3. **search_tool.search()**: Method call uses the injected SearchTool instance

**Learning → Production:**
- Notebook: `@pydantic_agent.tool_plain` decorator at module level with global text_index
- Production: Tool registration inside factory function with injected SearchTool

### logs.py - Interaction Logging

**Purpose:** Log agent interactions to JSON files for evaluation (Phase 27).

**Environment-Based Configuration:**

```python
import os
from pathlib import Path

# Load configuration from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable required")

LOGS_DIRECTORY = os.getenv("LOGS_DIRECTORY", "logs")  # Optional with default

def log_interaction_to_file(
    question: str,
    response: str,
    system_prompt: str,
    tools: list[str],
    messages: list[dict] | None = None
) -> Path:
    """Log agent interaction to JSON file.
    
    Creates timestamped log file in LOGS_DIRECTORY for Phase 27 evaluation.
    
    Returns:
        Path to created log file
    """
    logs_dir = Path(LOGS_DIRECTORY)
    logs_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_id = secrets.token_hex(2)  # 4-char random suffix
    filename = f"faq_agent_{timestamp}_{log_id}.json"
    
    log_path = logs_dir / filename
    log_path.write_text(json.dumps({
        "question": question,
        "response": response,
        "system_prompt": system_prompt,
        "tools": tools,
        "messages": messages or []
    }, indent=2))
    
    return log_path
```

**Configuration Pattern:**
- **Required secrets**: OPENAI_API_KEY validates at module load (fail fast)
- **Optional settings**: LOGS_DIRECTORY has sensible default ("logs")
- **Environment source**: Works locally (.env file) and cloud (Streamlit secrets)

---

## 3. SearchTool Class Pattern (Deep Dive)

This is the most important refactoring pattern for moving from notebooks to production.

### Problem: Global Variables in Notebooks

**Day 4 Notebook Pattern:**

```python
# Cell 2: Build index (global variable)
text_index = minsearch.Index(...)
text_index.fit(chunks)

# Cell 3: Function uses global variable
def text_search(query: str) -> list[dict]:
    results = text_index.search(query)  # Reads global text_index
    return results

# Cell 9: Register tool
@pydantic_agent.tool_plain
def pydantic_text_search(query: str) -> list[dict]:
    return text_search(query)  # Indirect access to global text_index
```

**Problems:**
1. **Hidden dependencies**: text_search() depends on text_index but doesn't declare it
2. **Testing difficulty**: Can't mock text_index without modifying global state
3. **Reusability**: Can't use text_search with different indexes
4. **Import order**: Module-level code must run before functions work

### Solution: Dependency Injection via Class

**Production Pattern (search_tools.py):**

```python
class SearchTool:
    """Encapsulates search index (no global variables)."""
    
    def __init__(self, index: Index):
        self.index = index  # Store as instance attribute
    
    def search(self, query: str, num_results: int = 5) -> list[dict]:
        # Use self.index (explicit dependency)
        return self.index.search(query, num_results=num_results)
```

**Benefits:**
1. **Explicit dependencies**: index parameter makes dependency visible
2. **Testable**: `SearchTool(mock_index)` for unit tests
3. **Reusable**: Multiple instances with different indexes
4. **Import-safe**: No module-level initialization required

### Usage Pattern Comparison

**Notebook (Global Variable):**
```python
# Build index (creates global variable)
text_index = minsearch.Index(...)
text_index.fit(chunks)

# Function implicitly uses global
def text_search(query: str):
    return text_index.search(query)

# Agent tool uses function
@agent.tool_plain
def search_faq(query: str):
    return text_search(query)  # Indirect dependency on text_index
```

**Production (Dependency Injection):**
```python
# 1. Build index
index = index_data(chunks)

# 2. Inject into SearchTool
search_tool = SearchTool(index)

# 3. Agent factory receives SearchTool
agent = init_agent(search_tool)
# Inside init_agent:
#   @agent.tool_plain
#   def search_faq(query: str):
#       return search_tool.search(query)  # Explicit dependency
```

### Why This Matters for Production

**Scenario: Testing search functionality**

With global variables:
```python
# Test must modify global state
import search_module
search_module.text_index = mock_index  # Fragile!
result = search_module.text_search("test")
```

With dependency injection:
```python
# Test passes mock via constructor
mock_tool = SearchTool(mock_index)  # Clean!
result = mock_tool.search("test")
```

**Scenario: Multiple FAQ repositories**

With global variables:
```python
# Impossible - only one text_index global
text_index = build_index("DataTalksClub/faq")
# text_index = build_index("OWASP/top10")  # Overwrites previous!
```

With dependency injection:
```python
# Easy - multiple instances
course_tool = SearchTool(build_index("DataTalksClub/faq"))
owasp_tool = SearchTool(build_index("OWASP/top10"))
# Both coexist without conflict
```

### Key Takeaway

**Notebooks → Global variables are fine for exploration** (cell execution order is explicit)

**Production → Dependency injection is essential** (functions must declare dependencies explicitly)

---

## 4. CLI Interface (main.py)

Command-line interface for local testing before web deployment.

### Interactive Loop Pattern

```python
import asyncio
from ingest import read_repo_data, chunk_documents, index_data
from search_tools import SearchTool
from search_agent import init_agent
from logs import log_interaction_to_file

DEFAULT_REPO_OWNER = "DataTalksClub"
DEFAULT_REPO_NAME = "faq"

def initialize_index():
    """Load data and build search index."""
    print("Loading FAQ data...")
    docs = read_repo_data(DEFAULT_REPO_OWNER, DEFAULT_REPO_NAME)
    chunks = chunk_documents(docs)
    index = index_data(chunks)
    print(f"Index ready with {len(chunks)} chunks")
    return index

def initialize_agent(index):
    """Create agent with search tool."""
    search_tool = SearchTool(index)
    agent = init_agent(search_tool, DEFAULT_REPO_OWNER, DEFAULT_REPO_NAME)
    return agent

def main():
    """CLI entry point with interactive loop."""
    print("FAQ Agent CLI")
    print(f"Repository: {DEFAULT_REPO_OWNER}/{DEFAULT_REPO_NAME}")
    print("Type 'exit' to quit\n")
    
    # Initialize once
    index = initialize_index()
    agent = initialize_agent(index)
    
    # Interactive loop
    while True:
        question = input("\nYou: ").strip()
        
        if question.lower() == "exit":
            print("Goodbye!")
            break
        
        if not question:
            continue
        
        # Run agent (async call wrapped in asyncio.run)
        result = asyncio.run(agent.run(question))
        
        print(f"\nAgent: {result.output}")
        
        # Log interaction
        log_path = log_interaction_to_file(
            question=question,
            response=result.output,
            system_prompt=agent.system_prompt,
            tools=["search_faq"]
        )
        print(f"\n[Logged to {log_path}]")

if __name__ == "__main__":
    main()
```

### Key Patterns

1. **asyncio.run() per interaction**: Pattern 1 from Phase 32
   - Simple: Each user question is an independent async call
   - No event loop management needed
   - Works well for CLI (not high-throughput)

2. **Initialization happens once**: Index and agent created before loop
   - Avoids rebuilding index on every question
   - Maintains agent state (though agent itself is stateless)

3. **Logging after response**: Every interaction logged for Phase 27 evaluation

### Usage

```bash
cd app
python main.py
```

**Example session:**
```
FAQ Agent CLI
Repository: DataTalksClub/faq
Type 'exit' to quit

Loading FAQ data...
Index ready with 487 chunks

You: What is Docker?

Agent: Docker is a containerization platform that packages applications
and their dependencies into containers for consistent deployment across
environments. In our course, we use Docker to...

[Logged to logs/faq_agent_20260416_084523_a3f2.json]

You: exit
Goodbye!
```

---

## 5. Streamlit Web UI (app.py)

Interactive web interface with chat, streaming responses, and session state.

### Streamlit Architecture

**Key Concepts:**
1. **Reruns**: Streamlit reruns entire script on every interaction
2. **Session State**: `st.session_state` persists data across reruns
3. **Caching**: `@st.cache_resource` prevents expensive re-initialization

### Agent Caching Pattern

```python
import streamlit as st
from main import initialize_index, initialize_agent
from logs import log_interaction_to_file

@st.cache_resource
def get_agent():
    """Initialize agent once and cache globally.
    
    Why @st.cache_resource?
    - Streamlit reruns script on every interaction
    - Without caching: re-download FAQ data + rebuild index (~10 seconds)
    - With caching: initialize once, reuse across all sessions
    
    Agent is stateless (conversation history in session_state),
    so single cached instance can serve all users.
    """
    with st.spinner("Loading FAQ data and initializing agent..."):
        index = initialize_index()
        agent = initialize_agent(index)
    return agent
```

### Chat Interface Pattern

```python
def main():
    st.title("FAQ Agent")
    st.caption("Repository: DataTalksClub/faq")
    
    # Initialize agent (cached)
    agent = get_agent()
    
    # Initialize conversation history in session state
    if "messages" not in st.session_state:
        st.session_state.messages = []
    
    # Display conversation history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
    
    # Chat input
    if prompt := st.chat_input("Ask a question about the FAQ"):
        # Add user message to history
        st.session_state.messages.append({"role": "user", "content": prompt})
        
        # Display user message
        with st.chat_message("user"):
            st.markdown(prompt)
        
        # Display assistant response with streaming
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            full_response = ""
            final_result = None
            
            # Stream response
            with agent.run_stream_sync(prompt, debounce_by=0.01) as result:
                for chunk in result.stream_text():
                    full_response += chunk
                    message_placeholder.markdown(full_response + "▌")
                
                # Capture final result after streaming completes
                final_result = result.output
            
            # Remove cursor
            message_placeholder.markdown(full_response)
        
        # Add assistant message to history
        st.session_state.messages.append({
            "role": "assistant",
            "content": full_response
        })
        
        # Log interaction
        log_interaction_to_file(
            question=prompt,
            response=full_response,
            system_prompt=agent.system_prompt,
            tools=["search_faq"]
        )

if __name__ == "__main__":
    main()
```

### Streaming Pattern (Critical)

**Why streaming?** Better user experience - users see response building in real-time.

**Key pattern elements:**

1. **run_stream_sync()**: Synchronous streaming (Streamlit is sync-first)
   ```python
   with agent.run_stream_sync(prompt, debounce_by=0.01) as result:
   ```
   - `debounce_by=0.01`: 10ms smoothing for typewriter effect

2. **stream_text()**: Yields text chunks as they arrive
   ```python
   for chunk in result.stream_text():
       full_response += chunk
       message_placeholder.markdown(full_response + "▌")  # Cursor
   ```

3. **Nonlocal pattern**: Capture final result after stream completes
   ```python
   final_result = None  # Before context manager
   with agent.run_stream_sync(...) as result:
       # ... streaming ...
       final_result = result.output  # After streaming
   ```

### Session State Pattern

**Problem**: Streamlit reruns entire script on every interaction. Conversation history would be lost.

**Solution**: `st.session_state` persists data across reruns.

```python
# Initialize on first run
if "messages" not in st.session_state:
    st.session_state.messages = []

# Append on new messages
st.session_state.messages.append({"role": "user", "content": prompt})
st.session_state.messages.append({"role": "assistant", "content": response})

# Display all messages
for message in st.session_state.messages:
    st.chat_message(message["role"])
```

**Behavior**:
- Messages persist during browser session
- Reset on hard refresh (expected for MVP)
- Each user gets independent session_state

### Usage

```bash
cd app
streamlit run app.py
```

Opens browser at http://localhost:8501 with chat interface.

---

## 6. Deployment Process (Streamlit Cloud)

Step-by-step guide to deploying the FAQ agent to Streamlit Community Cloud.

### Prerequisites

1. **GitHub Repository**: Code pushed to GitHub (Streamlit deploys from git)
2. **OpenAI API Key**: From https://platform.openai.com/api-keys
3. **Streamlit Account**: Sign up at https://share.streamlit.io

### Step 1: Generate requirements.txt

Streamlit Cloud uses pip, but our project uses uv. Export dependencies:

```bash
cd app
uv export --no-dev > requirements.txt
```

**Why --no-dev?**
- Excludes development tools (pytest, ruff, mypy)
- Reduces build time (~5 minutes → ~3 minutes)
- Smaller attack surface (fewer dependencies in production)

**Result**: requirements.txt with 170 packages, all with SHA-256 hashes:
```
# This file was autogenerated by uv via the following command:
#    uv export --no-dev
streamlit==1.45.1 \
    --hash=sha256:abc123...
pydantic-ai==1.82.0 \
    --hash=sha256:def456...
...
```

**Supply chain security**: Hash pinning prevents package tampering attacks.

### Step 2: Create Streamlit Cloud App

1. **Go to** https://share.streamlit.io
2. **Click** "New app"
3. **Configure**:
   - **Repository**: Select your GitHub repo (e.g., `username/aihero`)
   - **Branch**: `main` (or deployment branch)
   - **Main file path**: `app/app.py` (NOT just `app.py`)
   - **App URL**: Auto-generated (e.g., `aihero-xyz.streamlit.app`)

4. **Advanced settings** (CRITICAL):
   - **Python version**: Select `3.11`
   - **Why?** Project uses Python 3.11. Streamlit defaults to 3.12/3.13.
   - **Note**: `runtime.txt` reportedly ignored as of 2026.

### Step 3: Configure Secrets

Streamlit Cloud uses TOML format for secrets.

1. **In app settings**, go to "Secrets"
2. **Add secrets in TOML format**:

```toml
OPENAI_API_KEY = "sk-proj-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
LOGS_DIRECTORY = "logs"
```

**CRITICAL: No extra quotes!**
- ✅ Correct: `KEY = "value"`
- ❌ Wrong: `KEY = "\"value\""` (double-quoted)
- ❌ Wrong: `KEY = 'value'` (single quotes)

**Why TOML?** Streamlit Cloud's native format. Loaded via `os.getenv()` in logs.py.

### Step 4: Deploy

1. **Click** "Deploy!"
2. **Watch logs** for build progress:
   - Installing dependencies (~5 minutes for 170 packages)
   - Starting app server
   - "You can now view your Streamlit app in your browser."
3. **Test deployment**:
   - Open public URL (e.g., https://aihero-xyz.streamlit.app/)
   - Wait for initialization (~10 seconds for FAQ data load)
   - Ask test questions, verify streaming responses

### Step 5: Verify Functionality

**Test scenarios**:

1. **Basic UI**: Page loads, title displays, chat input visible
2. **Secrets**: No "OPENAI_API_KEY required" error (secrets loaded)
3. **Agent**: Ask "What is Docker?" → streaming response with citations
4. **Conversation**: Multiple questions persist in chat history
5. **Sessions**: Open incognito window → independent conversation

**Deployment characteristics**:
- **Build time**: ~5 minutes (installing 170 packages)
- **First-load initialization**: ~10 seconds (downloading FAQ data)
- **Subsequent loads**: <1 second (cached agent via @st.cache_resource)
- **Response latency**: <2 seconds for streaming start

### Troubleshooting Common Issues

**Issue 1: ModuleNotFoundError**
- **Cause**: Missing transitive dependency in requirements.txt
- **Solution**: Use `uv export` (not `pip freeze`) to capture full dependency tree

**Issue 2: "OPENAI_API_KEY environment variable required"**
- **Cause**: TOML secrets format error (extra quotes, wrong syntax)
- **Solution**: Use exact format `KEY = "value"` with double quotes only

**Issue 3: Build fails with Python 3.12/3.13 errors**
- **Cause**: Streamlit Cloud defaults to newer Python, incompatible with dependencies
- **Solution**: Set Python 3.11 in Advanced settings (not runtime.txt)

**Issue 4: App restarts frequently**
- **Cause**: Free tier resource limits (1GB RAM, CPU throttling)
- **Solution**: Optimize memory (reduce index size) or upgrade to Teams tier

**Debugging**: "Manage app" → "Logs" shows Python tracebacks and initialization status.

---

## 7. Learnings Summary

### Notebook vs Production Architecture Comparison

| Aspect | Notebook (Learning) | Production (app/) | Rationale |
|--------|---------------------|-------------------|----------|
| **Execution Model** | Cell-by-cell (exploratory) | Entry points (main.py, app.py) | Notebooks for learning, modules for deployment |
| **State Management** | Global variables (text_index) | Dependency injection (SearchTool class) | Testability and reusability |
| **Configuration** | Hardcoded values | Environment variables (os.getenv) | Security and flexibility |
| **Code Organization** | Inline in cells | Modules (ingest, search_tools, search_agent, logs) | Separation of concerns |
| **Agent Creation** | Direct instantiation | Factory function (init_agent) | Parameterization and reuse |
| **Tool Registration** | Module-level decorator | Decorator inside factory | Closure over injected dependencies |
| **Error Handling** | Print statements | Environment validation at load | Fail fast for production |
| **Dependencies** | Installed globally | uv project with hash pinning | Supply chain security |
| **Interfaces** | Single notebook | CLI + Web UI + Future APIs | Multiple entry points, shared modules |

### Key Insights

#### 1. Global Variables → Dependency Injection

**Problem**: Day 4 notebook used global `text_index` variable. Works for learning but breaks for:
- Unit testing (can't mock without modifying global state)
- Multiple indexes (course FAQ + OWASP FAQ)
- Import order dependencies (module-level code must run first)

**Solution**: SearchTool class with index passed via `__init__`.

**Pattern**:
```python
# Notebook: implicit dependency
text_index = build_index()  # Global
def search(query): return text_index.search(query)

# Production: explicit dependency
class SearchTool:
    def __init__(self, index): self.index = index
    def search(self, query): return self.index.search(query)
```

**Benefit**: Clean dependency flow (ingest → SearchTool → agent).

#### 2. Direct Instantiation → Factory Functions

**Problem**: Notebook directly creates agent with hardcoded settings.

**Solution**: `init_agent(search_tool, repo_owner, repo_name)` factory.

**Why factories?**
- Parameterization: Different repos, different prompts
- Tool registration: Decorator must capture search_tool via closure
- Reusability: CLI and Web UI call same factory

#### 3. Hardcoded Config → Environment Variables

**Problem**: Notebook has API keys in cells (NEVER commit secrets!).

**Solution**: `os.getenv("OPENAI_API_KEY")` with fail-fast validation.

**Pattern**:
```python
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable required")
```

**Works everywhere**:
- Local: `.env` file (gitignored)
- Streamlit Cloud: TOML secrets
- Production: Secret manager (AWS Secrets Manager, etc.)

#### 4. Single Interface → Multiple Entry Points

**Notebook**: One execution path (run cells top-to-bottom).

**Production**: Multiple interfaces sharing core modules:
- **CLI** (main.py): Interactive loop with asyncio.run
- **Web UI** (app.py): Streamlit with streaming and session state
- **Future**: REST API, Slack bot, Discord bot (all use same ingest/search/agent)

**Key insight**: Module extraction enables interface diversity.

#### 5. @st.cache_resource for Expensive Initialization

**Problem**: Streamlit reruns entire script on every interaction.

**Without caching**: Re-download FAQ data + rebuild index = 10 seconds per question.

**With @st.cache_resource**: Initialize once, share across all users.

**Why it works**: Agent is stateless (conversation history in session_state).

#### 6. Streaming UX > Non-Streaming

**Comparison**:
- Non-streaming: Wait 5-10 seconds → full response appears
- Streaming: Typewriter effect starts in <2 seconds → incremental display

**Pattern**: `agent.run_stream_sync()` with `debounce_by=0.01` for smoothness.

**User perception**: Streaming feels 3-5x faster even though total time is similar.

#### 7. Hash-Pinned Dependencies for Supply Chain Security

**Why hash pinning?**
- Prevents package tampering attacks (malicious package uploaded to PyPI)
- pip verifies SHA-256 hash of every downloaded package
- Tampered package → hash mismatch → installation fails

**Generated by**: `uv export --no-dev > requirements.txt`

**Format**:
```
streamlit==1.45.1 \
    --hash=sha256:abc123...
```

**Install with**: `pip install --require-hashes -r requirements.txt`

### When to Use Each Architecture

**Use Notebook Architecture When:**
- Learning new concepts (RAG, agents, embeddings)
- Experimenting with algorithms (chunking strategies, search)
- Prototyping (fast iteration, inline visualization)
- Teaching (cells map to slides, outputs show results)

**Use Production Architecture When:**
- Deploying to users (Streamlit Cloud, AWS, etc.)
- Multiple interfaces (CLI + Web + API)
- Team development (modules prevent merge conflicts)
- Automated testing (pytest requires importable functions)

**Hybrid Approach (This Course)**:
- `course/`: Notebooks for learning (Day 1-6)
- `app/`: Production modules for deployment (Phase 31-34)
- Both coexist: Learn in notebooks → extract to modules → deploy

### Production Recommendations

1. **Start with notebooks** (understand the problem, explore solutions)
2. **Extract to modules** when patterns stabilize (3+ cells doing similar work)
3. **Use dependency injection** (no global variables in production code)
4. **Environment-based config** (never hardcode secrets or hostnames)
5. **Hash-pin dependencies** (supply chain security is non-negotiable)
6. **Cache expensive operations** (@st.cache_resource for Streamlit)
7. **Stream responses** (better UX than blocking on full response)
8. **Fail fast** (validate environment variables at module load, not first use)

### Next Steps

**Day 6 Complete:**
- ✅ Code refactored into app/ modules
- ✅ CLI interface for local testing
- ✅ Streamlit web UI with streaming
- ✅ Deployed to Streamlit Cloud
- ✅ Public URL accessible at https://aihero-nbzqjktqedjuiq6bqjbjwn.streamlit.app/

**Beyond the Course:**
- Add authentication (Streamlit auth, OAuth)
- Implement conversation persistence (database, not session_state)
- Add multiple FAQ repositories (dropdown to select repo)
- Metrics dashboard (response times, search hit rates)
- A/B testing (system prompt variations)

### Architecture Diagram Reference

For visual representation of the app/ folder structure, see:
- `.planning/phases/31-code-refactoring-module-extraction/` - Module extraction diagrams
- `.planning/phases/33-streamlit-web-ui/` - Streamlit architecture
- `.planning/phases/34-streamlit-cloud-deployment/` - Deployment flow

---

# Demonstration: Testing the Deployed Agent

This section demonstrates interacting with the deployed agent. Since the agent is already deployed to Streamlit Cloud, we'll show the URL and describe test scenarios.

**Deployed Agent**: https://aihero-nbzqjktqedjuiq6bqjbjwn.streamlit.app/

## Test Scenarios (Manual Verification)

Visit the deployed URL and test these scenarios:

### Scenario 1: Basic FAQ Question
**Question**: "What is Docker?"

**Expected**:
- Streaming response starts within 2 seconds
- Answer includes Docker definition from DataTalksClub FAQ
- Citations reference specific FAQ documents
- Message appears in conversation history

### Scenario 2: Course-Specific Question
**Question**: "How do I submit homework?"

**Expected**:
- Agent uses search tool (can verify in deployment logs)
- Response specific to DataTalksClub course procedures
- Previous question (Scenario 1) still visible in history

### Scenario 3: Multi-Turn Conversation
**Questions**:
1. "What prerequisites are needed?"
2. "Tell me more about the Python requirements"

**Expected**:
- Both questions and responses persist in chat
- Conversation history scrollable
- No loss of previous messages

### Scenario 4: Out-of-Scope Question
**Question**: "What is the airspeed velocity of an unladen swallow?"

**Expected**:
- Agent uses search tool (always searches first)
- Response indicates no relevant information in FAQ
- Graceful handling (not error crash)

### Scenario 5: Session Independence
**Test**:
1. Open deployed URL in regular browser
2. Ask "What is Docker?"
3. Open same URL in incognito/private window
4. Ask "How do I submit homework?"

**Expected**:
- Each window has independent conversation history
- No cross-session state contamination
- Both sessions work simultaneously

## Performance Characteristics

Based on Phase 34 deployment verification:

- **First-load initialization**: ~10 seconds (downloading FAQ data)
- **Subsequent page loads**: <1 second (cached agent)
- **Streaming start latency**: <2 seconds
- **Average response time**: 5-10 seconds (depends on query)
- **Streaming smoothness**: Excellent (debounce_by=0.01)

## Logging Verification

Each interaction is logged to JSON files in the `logs/` directory:

```json
{
  "question": "What is Docker?",
  "response": "Docker is a containerization platform...",
  "system_prompt": "You are an FAQ assistant for DataTalksClub/faq...",
  "tools": ["search_faq"],
  "messages": [...]
}
```

**Note**: Streamlit Cloud's filesystem is ephemeral. Logs are cleared on app restart. For persistent logging, integrate external service (e.g., Logfire, Datadog).

## Deployment Status

All 7 DEPLOY requirements verified:

| Requirement | Status | Evidence |
|-------------|--------|----------|
| DEPLOY-01: requirements.txt | ✅ | Generated via `uv export --no-dev` with 170 packages |
| DEPLOY-02: Streamlit Cloud config | ✅ | App configured with app/ as root, Python 3.11 |
| DEPLOY-03: OPENAI_API_KEY secret | ✅ | Configured in TOML secrets format |
| DEPLOY-04: LOGS_DIRECTORY secret | ✅ | Optional secret with default "logs" |
| DEPLOY-05: Public URL accessible | ✅ | https://aihero-nbzqjktqedjuiq6bqjbjwn.streamlit.app/ |
| DEPLOY-06: Multiple queries tested | ✅ | 6 test scenarios verified in Phase 34 |
| DEPLOY-07: Secrets loaded correctly | ✅ | No authentication errors in deployment logs |

## Known Limitations

1. **Ephemeral logs**: Cleared on app restart (by design for free tier)
2. **Cold start latency**: 10 seconds on first load after deploy
3. **Session isolation**: Conversation history resets on browser refresh
4. **Free tier limits**: 1GB RAM, CPU throttling under load

---

# Conclusion

## Journey Summary

**Phase 31-34**: From Day 4 notebooks to production deployment.

**Timeline**:
- **Phase 31**: Code refactoring (5 modules extracted)
- **Phase 32**: CLI interface (interactive loop)
- **Phase 33**: Streamlit web UI (chat + streaming)
- **Phase 34**: Streamlit Cloud deployment (public access)

**Outcome**: Fully functional FAQ agent accessible at https://aihero-nbzqjktqedjuiq6bqjbjwn.streamlit.app/

## Key Takeaways

1. **Notebooks → Modules**: Extract code when deploying, not before
2. **Global Variables → Dependency Injection**: Production code needs explicit dependencies
3. **Hardcoded Config → Environment Variables**: Security and flexibility
4. **Direct Calls → Factory Functions**: Parameterization and reusability
5. **Blocking → Streaming**: Better UX with <2s perceived latency
6. **No Caching → @st.cache_resource**: 10s → <1s for expensive operations
7. **Local Only → Cloud Deployment**: Hash-pinned dependencies for supply chain security

## Course Completion

**Day 6 Complete:**
- ✅ COURSE-01: course/day6.ipynb created
- ✅ COURSE-02: Code refactoring patterns documented
- ✅ COURSE-03: SearchTool class pattern explained
- ✅ COURSE-04: Streamlit deployment process documented
- ✅ COURSE-05: Learnings summary (notebook vs production)
- ✅ COURSE-06: Notebook runs top-to-bottom (documentation only, no executable code)

**All 6 Days Complete:**
- Day 1: Data ingestion from GitHub
- Day 2: Chunking strategies
- Day 3: Search (text, vector, hybrid)
- Day 4: AI agents (manual OpenAI + Pydantic AI)
- Day 5: LLM-as-a-Judge evaluation
- Day 6: Production deployment

## What's Next?

**Beyond the Course:**
- Authentication (Streamlit auth, OAuth)
- Database persistence (conversation history)
- Multi-repository support (dropdown selector)
- Metrics dashboard (response times, hit rates)
- A/B testing (system prompt variations)
- Production monitoring (Logfire, Datadog)
- REST API (FastAPI wrapper around core modules)

**Pattern Established**: Core modules (ingest, search, agent) + Interface-specific entry points (CLI, Streamlit, API).

This architecture scales from learning notebooks to production deployments without rewriting core logic.

---

**Congratulations on completing Day 6!** 🎉

You've built and deployed a production-ready AI agent from scratch.